In [0]:
from pyspark.sql import functions as F

df_est = spark.table("workspace.prod_silver.silver_estabelecimentos")
df_emp = spark.table("workspace.prod_silver.silver_empresas")

df_longevidade = (
    df_est.alias("est")
    .join(df_emp.alias("emp"), "nr_cnpj_basico", "inner")
    .select(
        F.col("est.nr_cnpj_completo").alias("nr_cnpj"),
        F.col("est.nr_cnpj_basico"),
        F.col("emp.nm_razao_social"),
        F.col("est.dt_inicio_atividade"),
        F.col("est.dt_situacao_cadastral"),
        F.col("est.cd_situacao_cadastral"),
        F.col("est.cd_motivo_situacao_cadastral"),
        # Idade operacional: calculada até a data de baixa/situação ou até hoje (se ativa)
        F.when(
            F.col("est.cd_situacao_cadastral") == "02",
            F.round(F.datediff(F.current_date(), F.col("est.dt_inicio_atividade")) / 365.25, 2)
        ).otherwise(
            F.round(F.datediff(F.col("est.dt_situacao_cadastral"), F.col("est.dt_inicio_atividade")) / 365.25, 2)
        ).alias("nu_idade_empresa_anos"),
        # Flag de alteração cadastral recente (últimos 365 dias)
        F.when(
            F.datediff(F.current_date(), F.col("est.dt_situacao_cadastral")) <= 365,
            True
        ).otherwise(False).alias("fl_alteracao_situacao_recente"),
        # Presença de processo judicial/situação especial
        F.col("emp.vl_capital_social"),
        F.col("est.sg_uf"),
        F.current_timestamp().alias("dt_carga_gold")
    )
)

(
    df_longevidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sg_uf")
    .saveAsTable("workspace.prod_gold.gold_dp_risco_longevidade_empresarial")
)
print("-> workspace.prod_gold.gold_dp_risco_longevidade_empresarial gravada com sucesso.")

-> workspace.prod_gold.gold_dp_risco_longevidade_empresarial gravada com sucesso.
